# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keshav-geu/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane is Content Refresh, where the goal is to rank pages by how likely they are to be declining and therefore worth reviewing.

I use Logistic Regression as a simple and interpretable model, and Random Forest as the main model. Both models produce probabilities, which can be used as ranking scores instead of treating the task only as a yes/no prediction problem.

The proxy target remains the same as in my earlier framing: a page is labelled as declining when trend_direction == "down". The primary evaluation metric is Precision@50 because the practical objective is to make the top of the review queue useful.

I do not use trend_direction or trend_pct as model features because they are directly related to the target and could cause leakage.

In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

try:
    df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
except FileNotFoundError:
    !git clone https://github.com/keshav-geu/flyrank-internship.git
    %cd flyrank-internship
    df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df["target"] = (df["trend_direction"] == "down").astype(int)

print("\nTarget distribution:")
print(df["target"].value_counts())

print("\nDeclining-page base rate:")
print(round(df["target"].mean(), 4))


Rows: 30000
Columns: 44

Target distribution:
target
1    16262
0    13738
Name: count, dtype: int64

Declining-page base rate:
0.5421


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a grouped train/test split based on client_id. Pages belonging to the same client are kept together instead of being randomly divided between training and testing.

This gives a more honest evaluation because pages from the same client may share similar content and search-performance patterns. A random row split could make the test set too similar to the training data.

I use an 80/20 grouped split with random_state=42 so the result is reproducible. The same held-out test rows will be used to evaluate both the Week-4 baseline and the learned models.

In [7]:
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

model_df = df.dropna(subset=["client_id"]).copy()

X = model_df[features]
y = model_df["target"]
groups = model_df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(splitter.split(X, y, groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))
print("Train base rate:", round(y_train.mean(), 4))
print("Test base rate:", round(y_test.mean(), 4))

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0
Train base rate: 0.5501
Test base rate: 0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compare the learned models against the same hand-written baseline rule from Week 4. All methods are evaluated on the same held-out test set created above.

The baseline is used directly as a ranking score, while Logistic Regression and Random Forest are trained only on the training split and produce probabilities for the test rows.

I report Precision@20 and Precision@50. Precision@50 remains my primary metric because the goal is to make the top of the review queue useful.

In [8]:
def precision_at_k(y_true, scores, k=50):
    result = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    result = result.sort_values("score", ascending=False).head(k)

    return result["actual"].mean()


# Week-4 baseline
baseline_test = model_df.iloc[test_idx].copy()

baseline_test["baseline_score"] = (
    baseline_test["impressions_90d"].fillna(0) / 1000
    + baseline_test["content_age_days"].fillna(0) / 180
    - baseline_test["ctr"].fillna(0) * 10
)

baseline_scores = baseline_test["baseline_score"].values


# Logistic Regression
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

logistic_model.fit(X_train, y_train)

logistic_scores = logistic_model.predict_proba(X_test)[:, 1]


# Random Forest
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_scores = rf_model.predict_proba(X_test)[:, 1]


# Comparison table
results = pd.DataFrame({
    "Method": [
        "Base rate",
        "Week-4 baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "Precision@20": [
        y_test.mean(),
        precision_at_k(y_test, baseline_scores, 20),
        precision_at_k(y_test, logistic_scores, 20),
        precision_at_k(y_test, rf_scores, 20)
    ],
    "Precision@50": [
        y_test.mean(),
        precision_at_k(y_test, baseline_scores, 50),
        precision_at_k(y_test, logistic_scores, 50),
        precision_at_k(y_test, rf_scores, 50)
    ]
})

results["Precision@20"] = results["Precision@20"].round(3)
results["Precision@50"] = results["Precision@50"].round(3)

results

,Method,Precision@20,Precision@50
0,Base rate,0.511,0.511
1,Week-4 baseline,0.350,0.440
2,Logistic Regression,0.700,0.580
3,Random Forest,0.500,0.600


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest achieved the best Precision@50 at 0.60, improving over the Week-4 baseline score of 0.44. This means 30 of the top 50 pages ranked by the model were actually labelled as declining. However, 20 of the top 50 were false positives, so the ranking is useful for prioritisation but is not a perfect indicator that a page needs a refresh.

The strongest feature by permutation importance was impressions_90d, followed by content_age_days and ctr. This suggests that recent search visibility, the age of the content, and click-through behaviour contain useful signal for identifying declining pages. avg_position also provided some signal, while word_count and days_since_last_update had very small or negative permutation importance in this test split, suggesting they added little reliable predictive value to this model.

Looking at the false positives, some pages had characteristics that could reasonably make the model suspicious, such as older content, low CTR, or relatively weak average positions, even though their observed target was not declining. This shows that the model is learning useful patterns rather than a deterministic rule.

There is also a trade-off between the two learned models. Logistic Regression performed better at Precision@20, reaching 0.70 compared with 0.50 for Random Forest, while Random Forest performed better at the primary Precision@50 metric, reaching 0.60 compared with 0.58. I therefore keep Random Forest as the main model for this lane, while reporting both results rather than treating greater model complexity as automatically better.

In [9]:
analysis_df = model_df.iloc[test_idx].copy()

analysis_df["actual_decline"] = y_test.values
analysis_df["rf_score"] = rf_scores

analysis_df = analysis_df.sort_values(
    "rf_score",
    ascending=False
)

top50 = analysis_df.head(50).copy()

false_positives = top50[
    top50["actual_decline"] == 0
].copy()

print("False positives in top 50:", len(false_positives))

cols_to_show = [
    "content_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "rf_score",
    "actual_decline"
]

false_positives[cols_to_show].head(3)

False positives in top 50: 20


,content_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,rf_score,actual_decline
12472,content_884c401ce126,174,92,101,23.1,0.00,1615.0,0.817805,0
12332,content_4d9f36001f06,275,104,3369,13.2,0.03,1643.0,0.812062,0
14635,content_0cbc4d4326d5,271,104,330,25.0,0.00,1692.0,0.810625,0


In [10]:
perm = permutation_importance(
    rf_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="roc_auc"
)

importance_df = pd.DataFrame({
    "feature": features,
    "importance": perm.importances_mean
}).sort_values(
    "importance",
    ascending=False
)

importance_df

,feature,importance
2,impressions_90d,0.070454
0,content_age_days,0.028120
4,ctr,0.010636
3,avg_position,0.010503
5,word_count,-0.004186
1,days_since_last_update,-0.010228


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.